In [ ]:
import os
import gymnasium as gym
import numpy as np
import json
from collections import deque
from torch.utils.tensorboard import SummaryWriter

# ĐỔI THƯ VIỆN SANG SB3-CONTRIB
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList, BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv

from mahjong_env import MahjongEnv

class SlidingWindowStatsCallback(BaseCallback):
    def __init__(self, window_size=50, verbose=1, log_file="shanten_history_log.txt"):
        super(SlidingWindowStatsCallback, self).__init__(verbose)
        self.window_size = window_size
        
        # Các Buffer làm phẳng dữ liệu (Cửa sổ 50 ván)
        self.win_buffer = deque(maxlen=window_size)
        self.final_shanten_buffer = deque(maxlen=window_size) # Shanten lúc chốt ván
        self.mean_shanten_buffer = deque(maxlen=window_size)  # Trung bình Shanten ĐANG diễn ra trong ván
        self.steps_to_win_buffer = deque(maxlen=window_size)
        
        # Các mảng đếm tạm thời cho từng luồng (n_envs)
        self.ep_steps = None
        self.ep_sum_shanten = None # Lưu tổng Shanten cộng dồn của ván hiện tại
        
        self.iteration_count = 0 
        self.log_file = log_file
        
        with open(self.log_file, "w", encoding="utf-8") as f:
            f.write("=== BẮT ĐẦU GHI NHẬN LỊCH SỬ SHANTEN ===\n")

    def _on_training_start(self) -> None:
        n_envs = self.training_env.num_envs
        self.ep_steps = np.zeros(n_envs, dtype=int)
        self.ep_sum_shanten = np.zeros(n_envs, dtype=float)

    def _on_step(self) -> bool:
        self.ep_steps += 1
        
        for i, done in enumerate(self.locals['dones']):
            info = self.locals['infos'][i]
            current_step_shanten = info.get('shanten', 8)
            
            # Cộng dồn Shanten của bước này vào tổng của ván
            self.ep_sum_shanten[i] += current_step_shanten
            
            if done:
                # 1. Thu thập dữ liệu khi ván kết thúc
                final_shanten = current_step_shanten
                is_win = 1.0 if final_shanten == -1 else 0.0
                
                # Tính Mean Shanten của toàn bộ ván đấu này
                ep_mean_shanten = self.ep_sum_shanten[i] / max(self.ep_steps[i], 1)
                
                # Đưa vào cửa sổ trượt (Sliding Window)
                self.win_buffer.append(is_win)
                self.final_shanten_buffer.append(final_shanten)
                self.mean_shanten_buffer.append(ep_mean_shanten)
                
                if is_win == 1.0:
                    self.steps_to_win_buffer.append(self.ep_steps[i])

                # 2. Ghi file lịch sử Shanten
                if "episode_shanten_history" in info:
                    shanten_arr = info["episode_shanten_history"]
                    with open(self.log_file, "a", encoding="utf-8") as f:
                        status = "WIN " if is_win else "DRAW"
                        f.write(f"Iter {self.iteration_count + 1:04d} | {status} | {shanten_arr}\n")
                
                # Reset bộ đếm cho ván tiếp theo của luồng i
                self.ep_steps[i] = 0
                self.ep_sum_shanten[i] = 0.0
                
        return True

    def _on_rollout_end(self) -> None:
        self.iteration_count += 1
        
        # Bơm dữ liệu vào TensorBoard của SB3
        if len(self.win_buffer) > 0:
            win_rate = np.mean(self.win_buffer)
            mean_final_shanten = np.mean(self.final_shanten_buffer)
            mean_shanten = np.mean(self.mean_shanten_buffer) # Chỉ số bạn vừa yêu cầu
            
            self.logger.record("custom_metrics/win_rate", win_rate)
            self.logger.record("custom_metrics/mean_final_shanten", mean_final_shanten)
            self.logger.record("custom_metrics/mean_shanten", mean_shanten)
            
            if len(self.steps_to_win_buffer) > 0:
                mean_steps = np.mean(self.steps_to_win_buffer)
                self.logger.record("custom_metrics/mean_steps_to_win", mean_steps)
            else:
                mean_steps = 0.0
                
            # In Terminal
            if self.verbose > 0:
                total_wins = sum(self.win_buffer)
                print("\n" + "="*55)
                print(f"🔄 HOÀN TẤT ITERATION THỨ {self.iteration_count}")
                print(f"├─ Thống kê {len(self.win_buffer)} ván gần nhất (Window size = {self.window_size}):")
                print(f"├─ Tỷ lệ thắng     : {win_rate*100:.2f}% ({total_wins} ván thắng)")
                print(f"├─ Mean Final Shan : {mean_final_shanten:.2f} (Shanten lúc chốt ván)")
                print(f"├─ Mean Shanten    : {mean_shanten:.2f} (Sức khỏe bài cả ván)")
                if total_wins > 0:
                    print(f"└─ Tốc độ tới      : {mean_steps:.1f} bước/ván.")
                else:
                    print(f"└─ AI đang miệt mài học cách hạ Shanten...")
                print("="*55 + "\n")

class ShantenHistoryWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.shanten_history = []

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.shanten_history = [info.get("shanten", 8)]
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.shanten_history.append(info.get("shanten", 8))
        if terminated or truncated:
            info["episode_shanten_history"] = self.shanten_history.copy()
        return obs, reward, terminated, truncated, info

def make_masked_env():
    def _init():
        env = MahjongEnv()
        env = Monitor(env)
        env = ShantenHistoryWrapper(env)
        env = ActionMasker(env, mask_fn)
        return env
    return _init

def mask_fn(env: gym.Env) -> np.ndarray:
    return env.unwrapped._action_mask()

def main():
    n_envs = 8
    print(f"Khởi tạo {n_envs} môi trường chạy song song...\n")
    env = SubprocVecEnv([make_masked_env() for _ in range(n_envs)])

    model_path = ""
    tb_log_dir = "./logs/ppo_mahjong_tensorboard/"
    
    if os.path.exists(model_path) and model_path != "":
        model = MaskablePPO.load(model_path, env=env, tensorboard_log=tb_log_dir)
        
        # TẢI MODEL CŨ -> KHÔNG reset số bước
        do_reset = False 
    else:
        policy_kwargs = dict(net_arch=[512, 512, 256])

        model = MaskablePPO(
            "MlpPolicy", 
            env,
            policy_kwargs=policy_kwargs,
            device="auto",
            learning_rate=3e-4,
            n_steps=256, 
            batch_size=128, 
            tensorboard_log=tb_log_dir,  
            verbose=1,
        )
        # MODEL MỚI TOANH -> BẮT BUỘC PHẢI RESET để nó đẻ ra thư mục log
        do_reset = True

    # Cài đặt Callbacks
    stats_callback = SlidingWindowStatsCallback(window_size=50, verbose=1)
    
    # KÍCH HOẠT LẠI CHECKPOINT ĐỂ KHÔNG BỊ MẤT MODEL GIỮA CHỪNG
    checkpoint_callback = CheckpointCallback(
        save_freq=max(100_000 // n_envs, 1), 
        save_path='./models/checkpoints/',
        name_prefix='mahjong_model'
    )

    callback_list = CallbackList([stats_callback, checkpoint_callback])

    print("=== BẮT ĐẦU HUẤN LUYỆN ===")
    model.learn(
        total_timesteps=2_000_000, 
        progress_bar=False,
        callback=callback_list,  
        reset_num_timesteps=do_reset
    )

    os.makedirs("models", exist_ok=True)
    model.save("models/maskable_ppo_mahjong_final")
    env.close() 

    print("\n=== CHẠY THỬ VÀ XUẤT REPLAY JSON ===")
    test_env = MahjongEnv()
    test_env = ActionMasker(test_env, mask_fn)
    
    obs, info = test_env.reset()
    done = False
    step_count = 0

    replay_data = {
        "metadata": {
            "player": "AI_Agent_Maskable_PPO",
            "game_type": "1_vs_3_Dummies"
        },
        "initial_hand": obs[:34].astype(int).tolist(), 
        "timeline": []
    }

    while not done:
        step_count += 1
        action_masks = info.get("action_mask")
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
        
        obs, reward, terminated, truncated, info = test_env.step(action)
        done = terminated or truncated
        
        turn_log = {
            "turn": step_count,
            "agent_discarded": int(action),
            "dummies_discarded": info.get("dummy_discards", []),
            "agent_drew": info.get("agent_drawn", -1),
            "shanten_after": info["shanten"],
            "wall_left": info["wall_remaining"]
        }
        replay_data["timeline"].append(turn_log)
        
        print(f"Bước {step_count:2d} | Đánh: {action:2d} | Bốc: {info.get('agent_drawn', -1):2d} | Shanten: {info['shanten']} | {info['shanten_detail']['status']}")
        
    if terminated:
        print("=> AI đã TỚI BÀI!")
        replay_data["result"] = "Winning"
    elif truncated:
        print("=> Hòa (Hết bài).")
        replay_data["result"] = "Draw"

    with open("mahjong_replay.json", "w", encoding="utf-8") as f:
        json.dump(replay_data, f, indent=4, ensure_ascii=False)

    print("=> Toàn bộ diễn biến đã được lưu vào file 'mahjong_replay.json'!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import gymnasium as gym
import numpy as np
import json
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from mahjong_env import MahjongEnv

# 1. Định nghĩa lại hàm lấy Mask (phải giống hệt lúc train)
def mask_fn(env: gym.Env) -> np.ndarray:
    return env.unwrapped._action_mask()

def run_and_export_replay(model_path, output_file="mahjong_replay.json"):
    # 2. Khởi tạo môi trường và bọc Masker
    env = MahjongEnv()
    env = ActionMasker(env, mask_fn)

    # 3. Tải mô hình đã lưu
    if not os.path.exists(model_path):
        print(f"Lỗi: Không tìm thấy file model tại {model_path}")
        return

    print(f"Đang tải model từ: {model_path}...")
    model = MaskablePPO.load(model_path, env=env)

    # 4. Bắt đầu ván đấu
    obs, info = env.reset()
    done = False
    step_count = 0

    # Cấu trúc dữ liệu Replay
    replay_data = {
        "metadata": {
            "model_used": model_path,
            "player": "AI_Agent",
            "date_exported": "2026-04-22"
        },
        "initial_hand": obs[:34].astype(int).tolist(), 
        "timeline": []
    }

    print("\n=== AI BẮT ĐẦU ĐÁNH THỬ 1 VÁN ===")
    
    while not done:
        step_count += 1
        
        # Lấy mask và để AI dự đoán (deterministic=True để AI đánh chuẩn nhất, không bốc đồng)
        action_masks = info.get("action_mask")
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
        
        # Thực thi hành động
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # Lưu log lượt đi
        turn_log = {
            "turn": step_count,
            "agent_discarded": int(action),
            "dummies_discarded": info.get("dummy_discards", []),
            "agent_drew": info.get("agent_drawn", -1),
            "shanten_after": info["shanten"],
            "shanten_status": info["shanten_detail"]["status"],
            "reward_this_step": float(reward)
        }
        replay_data["timeline"].append(turn_log)
        
        # In nhanh ra màn hình để theo dõi
        print(f"Lượt {step_count:2d} | AI đánh: {action:2d} | Shanten: {info['shanten']} | Reward: {reward:.2f}")

    # 5. Kết luận ván đấu
    if terminated:
        print("\n🎉 KẾT QUẢ: AI ĐÃ TỚI BÀI (WIN)!")
        replay_data["result"] = "Winning"
    else:
        print("\n🤝 KẾT QUẢ: HÒA (DRAW).")
        replay_data["result"] = "Draw"

    # 6. Xuất ra file JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(replay_data, f, indent=4, ensure_ascii=False)
    
    print(f"\n=> Đã lưu diễn biến ván đấu vào file: {output_file}")

if __name__ == "__main__":
    # Thay đổi đường dẫn tới file model của bạn ở đây
    MODEL_FILE = "models/checkpoints_1/mahjong_model_1000000_steps.zip" 
    run_and_export_replay(MODEL_FILE)

In [ ]:
import os
import gymnasium as gym
import numpy as np
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from mahjong_env import MahjongEnv

# 1. Hàm lấy mask
def mask_fn(env: gym.Env) -> np.ndarray:
    return env.unwrapped._action_mask()

def evaluate_last_5_steps(model_path, num_episodes=100):
    # 2. Khởi tạo môi trường
    env = MahjongEnv()
    env = ActionMasker(env, mask_fn)

    if not os.path.exists(model_path):
        print(f"Lỗi: Không tìm thấy file model tại {model_path}")
        return

    print(f"Đang tải model từ: {model_path}...")
    model = MaskablePPO.load(model_path, env=env)

    # Dictionary để lưu tổng số lần xuất hiện của các mức Shanten trong 5 bước cuối
    # Mình theo dõi từ -1 (Tới bài) cho đến 6 (Rác)
    total_shanten_counts = {i: 0 for i in range(-1, 9)}
    
    print(f"\n🚀 Đang mô phỏng {num_episodes} ván đấu (Vui lòng đợi vài giây)...")

    # 3. Vòng lặp mô phỏng
    for ep in range(num_episodes):
        obs, info = env.reset()
        done = False
        
        # Mảng lưu trữ trạng thái Shanten qua từng bước của ván này
        # Bắt đầu bằng Shanten lúc vừa chia bài xong
        shanten_history = [info['shanten']] 

        while not done:
            action_masks = info.get("action_mask")
            action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
            
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            # Ghi nhận Shanten sau mỗi lượt đánh
            shanten_history.append(info['shanten'])

        # 4. Trích xuất 5 bước cuối cùng của ván đấu
        # Nếu ván đấu diễn ra nhanh hơn 5 bước, nó sẽ lấy toàn bộ ván
        last_5_steps = shanten_history[-5:]
        
        # Cộng dồn vào bộ đếm tổng
        for s in last_5_steps:
            if s in total_shanten_counts:
                total_shanten_counts[s] += 1

    # 5. Tính toán trung bình và in kết quả
    print("\n" + "="*50)
    print(f" BÁO CÁO: TẦN SUẤT SHANTEN TRONG 5 BƯỚC CUỐI ({num_episodes} VÁN) ")
    print("="*50)
    print("Mỗi ván có tối đa 5 bước cuối được xét. Dưới đây là\ntrung bình số lượt/ván mà AI đạt mức Shanten tương ứng:\n")

    # In đúng các Shanten từ 1 đến 6 theo yêu cầu
    for shanten_level in range(1, 7):
        # Tính trung bình: Tổng số lượt xuất hiện chia cho số ván
        average_per_episode = total_shanten_counts[shanten_level] / num_episodes
        print(f" 🔹 Shanten {shanten_level}: {average_per_episode:>4.2f} lần / ván")
        
    print("-" * 50)
    # Khuyến mãi thêm thông số của Tenpai và Tới bài để bạn xem tỷ lệ thắng
    avg_tenpai = total_shanten_counts[0] / num_episodes
    avg_win = total_shanten_counts[-1] / num_episodes
    print(f" 🎯 Shanten  0 (Tenpai) : {avg_tenpai:>4.2f} lần / ván")
    print(f" 🏆 Shanten -1 (Tới bài): {avg_win:>4.2f} lần / ván")
    print("="*50)

if __name__ == "__main__":
    # ĐỪNG QUÊN SỬA LẠI ĐƯỜNG DẪN MODEL NẾU CẦN
    MODEL_FILE = "models/checkpoints_1/mahjong_model_1000000_steps.zip" 
    
    # Chạy thử 100 ván
    evaluate_last_5_steps(MODEL_FILE, num_episodes=100)